Segmenting Consumers of Bath Soap - Module 5 - Samuel Scott

## Part 1: Business Situation
### The Agency
CRISA is an Asian market research agency that tracks consumer purchase behavior across consumer
goods, both durable and nondurable. In one major project it follows numerous product categories, such
as detergents, and dozens of brands within each. To track behavior, CRISA built household panels in
more than 100 cities and towns across India, covering most of the urban market. The households were
drawn by stratified sampling on socioeconomic status and market to keep the sample representative, and
a subset of 600 households is analyzed here.
CRISA keeps both transaction data (one row per transaction) and household data (one row per
household). For each household it maintains demographics (updated annually), possession of durable
goods (updated annually, from which an affluence index is computed), and purchase data by product
category and brand (updated monthly). Its clients fall into two groups: advertising agencies that subscribe
to the database and advise their own clients on advertising and promotion, and consumer goods
manufacturers that monitor their market share.

### The Segmentation Problem
CRISA has traditionally segmented markets by purchaser demographics. It now wants to segment on two
sets of variables tied more directly to the purchase process and to brand loyalty: purchase behavior
(volume, frequency, susceptibility to discounts, and brand loyalty) and basis of purchase (price and selling
proposition).
Better segmentation along these lines would reveal which demographic attributes go with which purchase
behaviors and degrees of loyalty, so promotion budgets could be deployed more effectively. It would let
CRISA's client, a firm called IMRB, design more cost-effective promotions aimed at the right segments,
run multiple promotions targeted at different segments at different times of year, and build more effective
customer reward systems to strengthen brand loyalty.

### Measuring Brand Loyalty
Several variables capture brand loyalty, and they capture different things. The number of distinct brands a
household buys is one measure. But a household that buys one or two brands briefly and then settles on
a third for a long stretch is not the same as one that constantly switches among three brands, so how
often a household switches (its brand runs) is a second measure. A third is concentration of spend: a
household that puts 90 percent of its money on a single brand is more loyal than one that spreads its
spending evenly across brands. All three can be derived from the purchase-summary and brand-share
variables, and how you combine them is one of the key modeling decisions in this case.

In [1]:
!pip install adjustText

In [2]:
import heapq
import random 
from collections import defaultdict
import matplotlib.pyplot as plt
import mlba
import pandas as pd
import surprise
from mlxtend.frequent_patterns import apriori, association_rules
from surprise import Dataset, Reader, KNNBasic
from surprise.model_selection import train_test_split
import numpy as np
import seaborn as sns
from adjustText import adjust_text
from pandas.plotting import parallel_coordinates
from scipy.cluster.hierarchy import dendrogram, fcluster, linkage
from sklearn import preprocessing
from sklearn.cluster import KMeans
from sklearn.metrics import pairwise

In [3]:
bathsoap = pd.read_csv('C:/Users/samsc/Desktop/ADS-505/BathSoapHousehold.csv')
bathsoap.head()

,Member id,SEC,FEH,MT,SEX,AGE,EDU,HS,CHILD,CS,...,PropCat 6,PropCat 7,PropCat 8,PropCat 9,PropCat 10,PropCat 11,PropCat 12,PropCat 13,PropCat 14,PropCat 15
0,1010010,4,3,10,1,4,4,2,4,1,...,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.028037,0.0,0.130841,0.339564
1,1010020,3,2,10,2,2,4,4,2,1,...,0.347048,0.026834,0.016100,0.014311,0.0,0.059034,0.000000,0.0,0.080501,0.000000
2,1014020,2,3,10,2,4,5,6,4,1,...,0.121212,0.033550,0.010823,0.008658,0.0,0.000000,0.016234,0.0,0.561688,0.003247
3,1014030,4,0,0,0,4,0,0,5,0,...,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.600000,0.000000
4,1014190,4,1,10,2,3,4,4,3,1,...,0.000000,0.000000,0.048193,0.000000,0.0,0.000000,0.000000,0.0,0.144578,0.000000


In [4]:
bathsoap.info()

<class 'pandas.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 46 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Member id              600 non-null    int64  
 1   SEC                    600 non-null    int64  
 2   FEH                    600 non-null    int64  
 3   MT                     600 non-null    int64  
 4   SEX                    600 non-null    int64  
 5   AGE                    600 non-null    int64  
 6   EDU                    600 non-null    int64  
 7   HS                     600 non-null    int64  
 8   CHILD                  600 non-null    int64  
 9   CS                     600 non-null    int64  
 10  Affluence Index        600 non-null    int64  
 11  No. of Brands          600 non-null    int64  
 12  Brand Runs             600 non-null    int64  
 13  Total Volume           600 non-null    int64  
 14  No. of  Trans          600 non-null    int64  
 15  Value            

### Question 1 - Segment the Households with k-Means
Use k-means clustering to identify clusters of households based on each of the following variable sets.

#### 1.1 Cluster on Purchase Behavior
Cluster households using the variables that describe purchase behavior, including brand loyalty.

In [5]:
loyalty = bathsoap[['Br. Cd. 57, 144', 'Br. Cd. 55', 'Br. Cd. 272','Br. Cd. 286', 'Br. Cd. 24', 'Br. Cd. 481', 'Br. Cd. 352', 'Br. Cd. 5']]
bathsoap['Loyalty'] = loyalty.max(axis=1)
purchase_behavior = bathsoap[['No. of Brands','Brand Runs', 'Total Volume', 'No. of  Trans', 'Value', 'Trans / Brand Runs', 'Vol/Tran', 'Avg. Price ',
                             'Pur Vol No Promo - %', 'Pur Vol Promo 6 %', 'Pur Vol Other Promo %', 'Loyalty']]
purchase_behavior_norm = purchase_behavior.apply(preprocessing.scale, axis=0)
kmeans_purch_behav = KMeans(n_clusters=3,random_state=1).fit(purchase_behavior_norm)
membpd.Series(kmeans_purch_behav.labels_



array([2, 1, 1, 2, 2, 1, 2, 2, 0, 2, 1, 1, 0, 2, 2, 1, 1, 2, 2, 1, 2, 2,
       0, 2, 2, 1, 2, 2, 2, 2, 2, 2, 2, 0, 2, 1, 2, 2, 2, 2, 2, 2, 0, 2,
       0, 2, 2, 2, 2, 1, 0, 2, 2, 0, 2, 0, 0, 0, 0, 2, 1, 0, 0, 2, 1, 2,
       1, 1, 2, 2, 2, 1, 2, 2, 2, 2, 2, 2, 0, 2, 2, 0, 2, 1, 2, 0, 1, 0,
       2, 2, 1, 0, 2, 1, 1, 1, 0, 1, 2, 2, 2, 2, 0, 1, 1, 1, 1, 2, 1, 1,
       2, 2, 2, 1, 0, 1, 1, 2, 1, 1, 1, 1, 2, 0, 0, 0, 0, 2, 2, 0, 1, 1,
       2, 1, 2, 1, 2, 1, 2, 0, 2, 2, 2, 2, 0, 2, 2, 1, 1, 1, 2, 2, 2, 2,
       1, 1, 1, 2, 2, 0, 2, 0, 0, 1, 0, 1, 0, 2, 1, 2, 1, 1, 2, 2, 0, 1,
       0, 2, 0, 2, 2, 0, 1, 1, 1, 2, 0, 1, 0, 1, 2, 1, 2, 2, 2, 1, 1, 1,
       1, 2, 2, 0, 0, 2, 2, 2, 2, 0, 2, 1, 2, 2, 1, 0, 0, 2, 2, 1, 2, 1,
       2, 0, 0, 2, 1, 2, 2, 1, 1, 2, 2, 0, 2, 0, 2, 0, 1, 2, 0, 2, 1, 1,
       1, 2, 2, 2, 2, 2, 2, 2, 2, 1, 2, 1, 2, 2, 2, 1, 0, 1, 2, 2, 1, 0,
       1, 2, 2, 2, 2, 1, 0, 0, 2, 1, 2, 2, 1, 1, 2, 2, 0, 2, 1, 0, 0, 1,
       1, 2, 1, 1, 1, 2, 2, 1, 2, 2, 2, 1, 1, 1, 1,

#### 1.2 Cluster on Basis of Purchase
Cluster households using the variables that describe the basis for purchase (price and selling
proposition).

In [6]:
basis_purchase = bathsoap[['Others 999','Pr Cat 1','Pr Cat 2','Pr Cat 3','Pr Cat 4','PropCat 5','PropCat 6', 'PropCat 7', 'PropCat 8', 
                          'PropCat 9', 'PropCat 10', 'PropCat 11', 'PropCat 12', 'PropCat 13', 'PropCat 14', 'PropCat 15']]
basis_purchase_norm = basis_purchase.apply(preprocessing.scale, axis=0)
kmeans_basis_purch = KMeans(n_clusters=3,random_state=1).fit(basis_purchase_norm)
kmeans_basis_purch.labels_

array([0, 0, 1, 1, 0, 0, 0, 1, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 1, 1,
       1, 1, 1, 2, 0, 0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1,
       0, 0, 0, 0, 1, 2, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1,
       1, 2, 0, 0, 0, 0, 1, 0, 0, 2, 0, 1, 0, 0, 0, 1, 1, 2, 0, 0, 0, 0,
       2, 1, 2, 0, 1, 2, 2, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0,
       1, 0, 0, 0, 0, 2, 0, 1, 0, 0, 0, 2, 0, 2, 0, 1, 0, 0, 0, 2, 0, 0,
       2, 0, 1, 2, 0, 0, 0, 0, 0, 1, 0, 1, 2, 1, 1, 0, 0, 0, 0, 0, 1, 1,
       0, 2, 2, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0,
       0, 1, 1, 0, 0, 0, 2, 2, 0, 0, 0, 0, 0, 2, 0, 2, 0, 0, 0, 0, 0, 0,
       2, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 2, 0, 2, 1, 2,
       0, 0, 1, 2, 0, 0, 0, 0, 2, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 2, 0, 0,
       0, 0, 1, 1, 1, 2, 2, 0, 0, 2, 0, 0, 0, 0, 0, 1, 0, 0, 2, 0, 0, 0,
       2, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 2, 0,
       0, 0, 0, 2, 2, 2, 0, 0, 0, 0, 1, 0, 0, 2, 0,

#### 1.3 Cluster on Both
Cluster households using both the purchase-behavior and basis-of-purchase variables together.

In [7]:
comb = pd.concat([purchase_behavior, basis_purchase], axis=1)
kmeans_comb = KMeans(n_clusters=3,random_state=1).fit(comb)
kmeans_comb.labels_

array([0, 2, 1, 0, 0, 2, 0, 0, 1, 0, 2, 2, 2, 2, 0, 2, 2, 2, 2, 0, 0, 0,
       1, 0, 0, 2, 2, 2, 2, 0, 2, 0, 0, 1, 2, 2, 0, 0, 2, 0, 0, 2, 1, 0,
       1, 0, 0, 0, 0, 2, 2, 2, 0, 1, 2, 2, 1, 2, 1, 0, 2, 2, 2, 2, 2, 0,
       1, 1, 0, 2, 0, 2, 2, 0, 2, 0, 0, 0, 2, 0, 0, 1, 2, 0, 0, 2, 2, 1,
       0, 2, 2, 1, 0, 0, 2, 2, 2, 2, 0, 2, 2, 2, 1, 2, 2, 2, 2, 2, 2, 0,
       2, 2, 2, 2, 2, 0, 2, 0, 2, 0, 2, 2, 2, 1, 1, 1, 1, 2, 0, 2, 2, 2,
       0, 0, 0, 0, 2, 0, 2, 1, 2, 2, 2, 2, 2, 2, 0, 0, 0, 0, 0, 0, 0, 2,
       2, 2, 0, 0, 0, 1, 2, 1, 1, 2, 1, 2, 1, 0, 2, 2, 2, 0, 0, 0, 1, 2,
       1, 2, 1, 2, 0, 2, 2, 0, 2, 2, 1, 2, 1, 2, 2, 2, 0, 2, 2, 0, 2, 0,
       2, 2, 2, 1, 1, 0, 0, 0, 0, 1, 2, 2, 2, 0, 2, 2, 1, 0, 2, 0, 2, 2,
       2, 1, 2, 0, 0, 0, 2, 2, 0, 0, 2, 1, 2, 1, 0, 2, 2, 2, 1, 0, 2, 0,
       0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2, 0, 2, 2, 1, 0, 0, 0, 0, 2,
       2, 2, 2, 2, 2, 0, 1, 1, 0, 0, 2, 2, 2, 2, 0, 0, 1, 2, 2, 1, 2, 0,
       0, 2, 2, 2, 2, 0, 0, 2, 0, 2, 2, 2, 2, 2, 0,

### Question 2 - Choose and Profile the Best Segmentation
From the three clustering solutions in Question 1, select the segmentation you think is most useful, and
comment on the characteristics of its clusters across three lenses: demographics, brand loyalty, and basis
of purchase. This profile is what guides the development of advertising and promotional campaigns, so
describe each segment in terms a campaign planner could act on.